<a href="https://colab.research.google.com/github/parshav42/learing/blob/main/medical_insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
!unzip -b /content/medical.zip -d /content/

In [ ]:
md = pa.read_csv('/content/medical_insurance.csv')

In [ ]:
md.head()

sex - 2 catg
region - 4 catg


In [ ]:
md.describe()

In [ ]:
#EDA

#check missing value

md.isnull().sum()


In [ ]:
md['region'].value_counts()

In [ ]:
md['sex'].describe()

In [ ]:
#find dubalicate rows
md.duplicated()
md[md.duplicated()]

In [ ]:
#drop dubalicate
md.drop_duplicates(inplace=True)

In [ ]:
md['age'].plot(kind ='kde')

In [ ]:
md['age'].plot(kind='box')

In [ ]:
md.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_columns = md.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(sparse_output=False)

encoded_data = encoder.fit_transform(md[categorical_columns])

encoded_df = pa.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_columns)
)

final_df = pa.concat(
    [md.drop(columns=categorical_columns), encoded_df],
    axis=1
)

print("\nOne-Hot Encoded Data:")
print(final_df)

In [ ]:
pa.crosstab(md['children'],md['sex'],normalize='columns')*100

In [ ]:
X = final_df.iloc[:,0:6]
y = final_df.iloc[:,-1]


In [ ]:
X_train , X_test, y_train,y_test = train_test_split(X,y,test_size =0.2)
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

lr = make_pipeline(
    SimpleImputer(strategy='median'),
    LinearRegression())

lr.fit(X_train,y_train)


In [ ]:
#Generate predictions.
predictions = lr.predict(X_test)
print(predictions)

In [ ]:
mask = ~np.isnan(y_test) & ~np.isnan(predictions)
y_test = y_test[mask]

In [ ]:
y_test.dropna()

In [ ]:
np.delete(predictions, -1, axis=0)

In [ ]:
y_test.shape,predictions.shape

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, predictions)

# 2. Mean Squared Error (MSE)
mse = mean_squared_error(y_test,predictions)

# 3. Root Mean Squared Error (RMSE)
# rmse = mean_squared_error(y_test, predictions, squared=False)


# 4. R-squared (R²) Score
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
# print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np # Ensure numpy is imported for operations like astype

# Convert continuous predictions to binary labels using a threshold
predictions_binary = (predictions > 0.5).astype(int)

accuracy_score(y_test, predictions_binary)

In [ ]:
error = abs(y_test - predictions)

plt.scatter(y_test, predictions, c=error, cmap='viridis', alpha=0.8)
plt.colorbar(label='Absolute Error')
plt.show()
# plt.scatter(y_test, predictions,color='red')
# plt.xlabel('Actual Values (y_test)')
# plt.ylabel('Predicted Values')
# plt.title('Actual vs. Predicted Values')
# plt.show()
# sn.scatterplot(X='y_test',y='predictions')

In [ ]:
import joblib

joblib.dump(lr, 'model_Student_performance.pkl')